# Convolution 2.0

## Import libraries

In [130]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypher
import warnings
from astropy.io import fits
from astropy.modeling import models, fitting
from astropy.utils.exceptions import AstropyWarning
from pathlib import Path



## Set directories

Determine file paths and obtain lists of galaxy images and PSF files for each survey inside Input/

In [77]:
CWD = Path.cwd()
ROOT = CWD.parents[1]
ASTROVELLO_DIR = ROOT / "AsTrovello_2_0"

input_dir = ASTROVELLO_DIR / "Input"

survey_paths = list(input_dir.glob("*"))
survey_names = [f.name for f in survey_paths]

galaxy = "ngc1097"

survey_names

['PHANGS', 'S4G']

## Obtain all survey files (Science images and PSFs)

In [78]:
image_files = []
psf_files = []
for survey in survey_names:
    image_dir = input_dir / survey / "galaxies" / galaxy
    psf_dir = input_dir / survey / "PSF"

    if survey == "PHANGS":
        current_image_files = list(image_dir.glob("*_exp-drc-sci.fits"))
        current_psf_files = list(psf_dir.glob("*PSFSTD*.fits"))
    elif survey == "S4G":
        current_image_files = list(image_dir.glob(f"{galaxy.upper()}.phot.*.fits"))
        current_psf_files = list(psf_dir.glob("*_col129_row129.fits"))

    image_files = image_files + current_image_files
    psf_files = psf_files + current_psf_files


In [79]:
image_files

[WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/PHANGS/galaxies/ngc1097/hlsp_phangs-hst_hst_wfc3-uvis_ngc1097mosaic_f555w_v1_exp-drc-sci.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/PHANGS/galaxies/ngc1097/hlsp_phangs-hst_hst_wfc3-uvis_ngc1097mosaic_f814w_v1_exp-drc-sci.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/S4G/galaxies/ngc1097/NGC1097.phot.1.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/S4G/galaxies/ngc1097/NGC1097.phot.2.fits')]

In [80]:
psf_files

[WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/PHANGS/PSF/PSFSTD_WFC3UV_F275W.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/PHANGS/PSF/PSFSTD_WFC3UV_F336W.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/PHANGS/PSF/PSFSTD_WFC3UV_F438W.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/PHANGS/PSF/PSFSTD_WFC3UV_F555W.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/PHANGS/PSF/PSFSTD_WFC3UV_F814W.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/S4G/PSF/IRAC1_col129_row129.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/S4G/PSF/IRAC2_col129_row129.fits')]

## Determine PSF resolutions 
Calculate FWHM to determine each surveys resolution and define the master file for convolution (lowest resolution).

### 1. Clean PSFs

In [81]:
phangs_img_header = fits.getheader(image_files[2], 0) 
phangs_img_header["INSTRUME"]

'IRAC'

In [82]:
SURVEY_CONFIG = {
                    "PHANGS": 
                    {
                        "TELESCOP": "HST",
                        "INSTRUME": "WFC3",
                        "pixel_scale_arcsec": 0.0395,
                        "binned_factor": 4,
                        "unit_type": "electrons/s", # usado em units.py
                        "force_tan_sip": False
                    },
                    "S4G":
                    {
                        "TELESCOP": "Spitzer",
                        "INSTRUME": "IRAC",
                        "pixel_scale_arcsec": 
                        {
                            1: 1.221, # Channel 1
                            2: 1.223 # Channel 2
                        },
                        "binned_factor": 5,
                        "unit_type": "mjy/sr", # usado em units.py
                        "force_tan_sip": True
                    }
                }

In [83]:
supported_instruments = [survey_data["INSTRUME"] for survey_data in SURVEY_CONFIG.values()]
psf_files[0].name

'PSFSTD_WFC3UV_F275W.fits'

In [120]:
def get_fwhm_simple(data):
    """
    Estimates the FWHM using a 2D Gaussian fit.
    Robust against background noise, negative pixels, and large image boxes.
    """
    # 1. Remove qualquer NaN que possa quebrar o algoritmo
    data_clean = np.nan_to_num(data, nan=0.0)
    
    # 2. Cria uma malha de coordenadas X e Y do mesmo tamanho da imagem
    y, x = np.mgrid[:data_clean.shape[0], :data_clean.shape[1]]
    
    # 3. Estima os parâmetros iniciais (chutes) para ajudar o algoritmo a convergir mais rápido
    max_val = np.max(data_clean)
    y_center, x_center = np.unravel_index(np.argmax(data_clean), data_clean.shape)
    
    # Cria o modelo inicial da Gaussiana
    g_init = models.Gaussian2D(amplitude=max_val, x_mean=x_center, y_mean=y_center, 
                               x_stddev=2.0, y_stddev=2.0)
    
    # 4. Inicializa o algoritmo de ajuste (Levenberg-Marquardt Mínimos Quadrados)
    fit_g = fitting.LevMarLSQFitter()
    
    # 5. Ajusta o modelo aos dados
    with warnings.catch_warnings():
        # Ignora avisos inofensivos do astropy caso a PSF seja muito ruidosa
        warnings.simplefilter('ignore')
        g_fit = fit_g(g_init, x, y, data_clean)
    
    # 6. Extrai o desvio padrão (sigma) do eixo X e Y e tira a média geométrica
    # A média geométrica lida melhor com PSFs ligeiramente elípticas
    sigma_eff = np.sqrt(abs(g_fit.x_stddev.value * g_fit.y_stddev.value))
    
    # 7. Converte Sigma para FWHM em pixels
    fwhm_pixels = 2.3548 * sigma_eff
    
    return fwhm_pixels
    
    return 2.355 * sigma

def calculaFWHM_radial_profile(file_list):
    """
    Iterates through a folder of PSF files, filters them by survey,
    and returns dictionaries containing their FWHM.
    """
    FWHM_dict, valid_files = {}, []
    #file_list = list(files_path.glob('*.fits'))
    
    for file in file_list:
        # Survey-specific filename parsing
        if 'S4G' in str(file_list):
            if file.name == 'IRAC1_col129_row129.fits': filter_name = 'irac1'
            elif file.name == 'IRAC2_col129_row129.fits': filter_name = 'irac2'
            else: continue
        if 'PHANGS' in str(file_list):
            filter_name = file.name.replace('.fits', '').split('_')[-1].lower() 
        else: continue

        try:
            with fits.open(file, ignore_missing_end=True) as hdu:
                # Dynamically find the data HDU
                data = next((h.data for h in hdu if h.data is not None), None)
                if data is not None:
                    if data.ndim == 3: data = data[0] # Flatten 3D PSF cubes
                    
                    FWHM_dict[filter_name] = get_fwhm_simple(data)
                    valid_files.append(file.name)
                    print(f"Succesfully read: {filter_name}")
        except Exception as e:
            print(f"Processing error {file.name}: {e}")

    return FWHM_dict, valid_files

def final_clean_psf(input_file, output_file):
    """
    Standardizes PSF headers and performs true downsampling for PyPHER compatibility.
    Calculates pixel scales, bins down oversampled PSFs, and ensures correct 
    centering and coordinate keywords.
    """
    if 'WFC3UV' in input_file:
        # HST scale: native 0.0395"/pix. We will bin down the 4x oversampled data.
        pixel_scale_arcsec = 0.0395 
        pixel_scale_deg = pixel_scale_arcsec / 3600.0

        with fits.open(input_file, ignore_missing_end=True) as hdu:
            # Average the PSF cube to get a 2D representative PSF
            data_2d = np.mean(hdu[0].data, axis=0)
            
            # Downsample the array by a factor of 4 (summing blocks of 4x4 pixels)
            data_2d_binned = block_reduce(data_2d, block_size=4, func=np.sum)
            
            # Force odd parity: PyPHER prefers kernels/PSFs with an odd number of pixels
            if data_2d_binned.shape[0] % 2 == 0:
                data_2d_binned = data_2d_binned[:-1, :-1]
                
            # Normalize to ensure flux conservation
            data_2d_binned = data_2d_binned / np.sum(data_2d_binned)

            new_hdu = fits.PrimaryHDU(data_2d_binned)
            
            # Inject WCS keywords required by PyPHER/Astropy
            new_hdu.header.update({
                'CTYPE1': 'RA---TAN', 'CTYPE2': 'DEC--TAN',
                'CRVAL1': 0.0, 'CRVAL2': 0.0,
                'CRPIX1': (data_2d_binned.shape[1] // 2) + 1, 'CRPIX2': (data_2d_binned.shape[0] // 2) + 1,
                'CDELT1': -pixel_scale_deg, 'CDELT2': pixel_scale_deg,
                'PIXSCALE': pixel_scale_arcsec
            })
            new_hdu.writeto(output_file, overwrite=True)
            print(f"==> File ready to be applied in PyPHER (Binned 4x): {os.path.basename(output_file)}")

    elif any(x in input_file for x in ['IRAC1', 'IRAC2']):
        # Spitzer scale: native ~1.22"/pix. We will bin down the 5x oversampled data.
        pixel_scale_arcsec = 1.221 if 'IRAC1' in input_file else 1.213
        pixel_scale_deg = pixel_scale_arcsec / 3600.0

        with fits.open(input_file) as hdu:
            data_raw = hdu[0].data
            
            # Downsample the array by a factor of 5
            data_2d_binned = block_reduce(data_raw, block_size=5, func=np.sum)
            
            # Force odd parity
            if data_2d_binned.shape[0] % 2 == 0:
                data_2d_binned = data_2d_binned[:-1, :-1]
                
            # Normalize to ensure flux conservation
            data_2d_binned = data_2d_binned / np.sum(data_2d_binned)

            new_hdu = fits.PrimaryHDU(data_2d_binned)
            new_hdu.header.update({
                'CTYPE1': 'RA---TAN', 'CTYPE2': 'DEC--TAN',
                'CRVAL1': 0.0, 'CRVAL2': 0.0,
                'CRPIX1': (data_2d_binned.shape[1] // 2) + 1, 'CRPIX2': (data_2d_binned.shape[0] // 2) + 1,
                'CDELT1': -pixel_scale_deg, 'CDELT2': pixel_scale_deg,
                'PIXSCALE': pixel_scale_arcsec
            })
            new_hdu.writeto(output_file, overwrite=True)
            print(f"==> File ready to be applied in PyPHER (Binned 5x): {os.path.basename(output_file)}")


In [129]:
def calculateFWHM(file_list, SURVEY_CONFIG):
    """
    AsTrovello 2.0
    @brief Iterates through a folder of PSF files, filters them by survey,
    and returns dictionaries containing their physical FWHM (in arcsec).
    """
    FWHM_dict, valid_files = {}, []
    
    # Silencia os avisos chatos de cabeçalho do Astropy
    warnings.simplefilter('ignore', category=AstropyWarning)
    
    for file in file_list:
        if file.name.startswith('.'):
            continue

        if 'S4G' in str(file):
            binned_factor = SURVEY_CONFIG["S4G"]["binned_factor"]
            if file.name == 'IRAC1_col129_row129.fits': 
                filter_name = 'irac1'
                pixscale = SURVEY_CONFIG["S4G"]["pixel_scale_arcsec"][1] / binned_factor
            elif file.name == 'IRAC2_col129_row129.fits': 
                filter_name = 'irac2'
                pixscale = SURVEY_CONFIG["S4G"]["pixel_scale_arcsec"][2] / binned_factor
            else: 
                continue
                
        elif 'PHANGS' in str(file):
            filter_name = file.name.replace('.fits', '').split('_')[-1].lower() 
            binned_factor = SURVEY_CONFIG["PHANGS"]["binned_factor"]
            pixscale = SURVEY_CONFIG["PHANGS"]["pixel_scale_arcsec"] / binned_factor
        else: 
            continue

        try:
            with fits.open(file, ignore_missing_end=True, ignore_missing_simple=True) as hdu:
                data = next((h.data for h in hdu if h.data is not None), None)
                
                if data is not None:
                    if data.ndim == 3: 
                        data = np.mean(data, axis=0) 
                    
                    # Usa o ajuste Gaussiano (ou a função robusta) que retorna em pixels
                    fwhm_pixels = get_fwhm_simple(data)
                    
                    # Converte para escala física (arcsec) usando a escala de pixel superamostrada
                    FWHM_dict[filter_name] = fwhm_pixels * pixscale
                    
                    valid_files.append(file.name)
                    print(f"Successfully read: {filter_name} (FWHM: {FWHM_dict[filter_name]:.4f} arcsec)")
                    
        except Exception as e:
            print(f"Processing error {file.name}: {e}")
            
    # Restaura os avisos para o resto do seu código
    warnings.simplefilter('default', category=AstropyWarning)
    
    return FWHM_dict, valid_files

In [126]:
psf_files

[WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/PHANGS/PSF/PSFSTD_WFC3UV_F275W.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/PHANGS/PSF/PSFSTD_WFC3UV_F336W.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/PHANGS/PSF/PSFSTD_WFC3UV_F438W.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/PHANGS/PSF/PSFSTD_WFC3UV_F555W.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/PHANGS/PSF/PSFSTD_WFC3UV_F814W.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/S4G/PSF/IRAC1_col129_row129.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/S4G/PSF/IRAC2_col129_row129.fits')]

In [127]:
fwhm_dict, valid_files = calculaFWHM_radial_profile(psf_files)

Succesfully read: row129
Succesfully read: row129


In [128]:
fwhm_dict, valid_files = calculateFWHM(psf_files, SURVEY_CONFIG)

Successfully read: f275w (FWHM: 0.0766 arcsec)
Successfully read: f336w (FWHM: 0.0810 arcsec)
Successfully read: f438w (FWHM: 0.0838 arcsec)
Successfully read: f555w (FWHM: 0.0825 arcsec)
Successfully read: f814w (FWHM: 0.0790 arcsec)
Successfully read: irac1 (FWHM: 1.5620 arcsec)
Successfully read: irac2 (FWHM: 1.5253 arcsec)


In [89]:
psf_files[0]

WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/PHANGS/PSF/PSFSTD_WFC3UV_F275W.fits')

In [98]:
from astropy.io import fits
import numpy as np

with fits.open(psf_files[1], ignore_missing_end=True, lazy_load_hdus=True) as hdu:
    hdu[0].verify('silentfix')     # verifica só a HDU 0, não a lista inteira
    header = hdu[0].header
    psf_cube = hdu[0].data         # shape (56, 101, 101)

In [108]:
psf_cube = fits.getdata(psf_files[1], ext=0, ignore_missing_end=True)

In [117]:
psf_cube.sum()

np.float32(726.35284)

In [91]:
fwhm_dict

{'f275w': np.float64(nan),
 'f336w': np.float64(nan),
 'f438w': np.float64(nan),
 'f555w': np.float64(nan),
 'f814w': np.float64(nan),
 'irac1': np.float64(43.00025444275528),
 'irac2': np.float64(46.6778234797745)}